# 📡 Notebook 06 — Model Monitoring & Data Drift Detection

**Goal:** Detect data drift (Population Stability Index) and monitor model performance over time. Supports SR 11-7 / Model Risk Management compliance.

> **Run time:** ~5 min

In [ ]:
# Import analysis and plotting libraries used for monitoring drift and model health
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
# Suppress noisy warnings so the monitoring output stays focused on key signals
warnings.filterwarnings('ignore')

# Load Feature Store and Scores
# Load the feature store and scored output tables needed for drift and performance checks
df_features = spark.table('silver_credit_risk_features').toPandas()
df_scores   = spark.table('gold_credit_risk_scores').toPandas()

# Confirm the monitoring tables contain the expected number of accounts
print(f'Features: {len(df_features)} accounts')
print(f'Scores:   {len(df_scores)} accounts')


## Step 1 — Population Stability Index (PSI) for CreditScore

PSI measures how much the distribution of a variable has shifted between training and scoring populations.

- **PSI < 0.1**: No significant shift ✅
- **0.1 ≤ PSI < 0.25**: Some shift ⚠️ — monitor
- **PSI ≥ 0.25**: Significant shift ❌ — retrain model

In [ ]:
# Define a reusable Population Stability Index helper for feature drift detection
def calculate_psi(expected, actual, buckets=10):
    """Calculate Population Stability Index between two distributions."""
    # Create percentile-based bins from the reference population
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints)
    breakpoints[0]  = -np.inf
    breakpoints[-1] =  np.inf

    # Count how many observations fall into each PSI bucket for both populations
    exp_counts = np.histogram(expected, bins=breakpoints)[0]
    act_counts = np.histogram(actual,   bins=breakpoints)[0]

    # Convert bucket counts to percentages and smooth any zero-count buckets
    exp_pct = np.where(exp_counts == 0, 0.001, exp_counts / len(expected))
    act_pct = np.where(act_counts == 0, 0.001, act_counts / len(actual))

    # Compute the PSI contribution of each bucket and sum them into one drift score
    psi_values = (act_pct - exp_pct) * np.log(act_pct / exp_pct)
    return np.sum(psi_values)

# Split into training (first 80%) and scoring (last 20%) as proxy
# Split the dataset into reference and scoring windows as a proxy for temporal drift monitoring
n_train = int(len(df_features) * 0.8)
train_scores_dist = df_features['CreditScore'].iloc[:n_train]
score_scores_dist = df_features['CreditScore'].iloc[n_train:]

# Calculate PSI for credit score to gauge whether the incoming population shifted materially
psi_credit = calculate_psi(train_scores_dist, score_scores_dist)

# Map the PSI score to stable, monitor, or retrain thresholds used in governance
status = ('✅ Stable' if psi_credit < 0.1
          else '⚠️ Monitor' if psi_credit < 0.25
          else '❌ Retrain Required')
print(f'PSI (CreditScore): {psi_credit:.4f}  →  {status}')


## Step 2 — PSI for All Numeric Features

In [ ]:
# Select the key numeric features that will be monitored for distribution drift
numeric_features = [
    'CreditScore','AnnualIncome','LoanAmount','DebtToIncomeRatio',
    'NumDelinquencies','EmploymentYears','NumOpenAccounts'
]

# Calculate PSI for each monitored feature and collect the results in a summary table
psi_results = []
# Compare the reference and scoring distributions feature by feature
for feat in numeric_features:
    train_vals = df_features[feat].iloc[:n_train].dropna()
    score_vals = df_features[feat].iloc[n_train:].dropna()
    psi_val    = calculate_psi(train_vals, score_vals)
    flag       = ('✅' if psi_val < 0.1 else '⚠️' if psi_val < 0.25 else '❌')
    psi_results.append({'Feature': feat, 'PSI': round(psi_val, 4), 'Status': flag})

# Convert the PSI results into a ranked dataframe for reporting
psi_df = pd.DataFrame(psi_results).sort_values('PSI', ascending=False)
print('Feature Drift Summary:')
print(psi_df.to_string(index=False))

# Plot
# Visualize feature drift severity against the monitor and retrain thresholds
plt.figure(figsize=(10,5))
colors = ['red' if p >= 0.25 else 'orange' if p >= 0.1 else 'green'
          for p in psi_df['PSI']]
plt.bar(psi_df['Feature'], psi_df['PSI'], color=colors)
plt.axhline(0.1, color='orange', linestyle='--', label='Monitor threshold (0.1)')
plt.axhline(0.25, color='red',    linestyle='--', label='Retrain threshold (0.25)')
plt.xticks(rotation=30, ha='right'); plt.ylabel('PSI')
plt.title('Population Stability Index by Feature')
plt.legend(); plt.tight_layout(); plt.show()


## Step 3 — Score Distribution Monitoring

In [ ]:
# Create side-by-side plots for score distribution and deployed risk-band mix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Default probability distribution
# Plot the distribution of predicted default probabilities across the scored portfolio
axes[0].hist(df_scores['DefaultProbability'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Default Probability'); axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Default Probability Scores')

# Risk category breakdown
# Summarize and plot how accounts are distributed across risk categories
risk_counts = df_scores['RiskCategory'].value_counts()
axes[1].pie(risk_counts, labels=risk_counts.index,
            colors=['green','yellow','orange','red'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Risk Category Distribution')

# Render the monitoring dashboard charts with readable spacing
plt.tight_layout(); plt.show()


## Step 4 — Model Performance Metrics (Cumulative)

In [ ]:
# Import the ranking metrics used to monitor production scoring quality
from sklearn.metrics import roc_auc_score, average_precision_score

# Compute performance metrics from realized defaults and predicted probabilities
auc  = roc_auc_score(df_scores['ActualDefault'], df_scores['DefaultProbability'])
ap   = average_precision_score(df_scores['ActualDefault'], df_scores['DefaultProbability'])

print('═' * 50)
# Print a compact performance dashboard for the latest scored population
print('  CREDIT RISK MODEL PERFORMANCE DASHBOARD')
print('═' * 50)
print(f'  Total Accounts Scored:  {len(df_scores)}')
print(f'  AUC-ROC:                {auc:.4f}')
print(f'  Avg Precision (AP):     {ap:.4f}')
print(f'  Actual Default Rate:    {df_scores["ActualDefault"].mean():.1%}')
print(f'  Avg Predicted Prob:     {df_scores["DefaultProbability"].mean():.1%}')
print('─' * 50)
print('  Risk Category Breakdown:')
# Break out the number of accounts assigned to each deployed risk category
for cat, cnt in df_scores['RiskCategory'].value_counts().sort_index().items():
    pct = cnt / len(df_scores) * 100
    print(f'    {cat:<18} {cnt:>4} accounts  ({pct:.1f}%)')
print('═' * 50)


## Step 5 — SR 11-7 Compliance Checklist

In [ ]:
# Print compliance checklist for model risk management
# Define the SR 11-7 model risk management checklist for this credit risk demo
checklist = [
    ('Model Documentation',        '✅', 'README.md + DEMO_SCRIPT.md created'),
    ('Model Validation',           '✅', '5-fold CV + held-out test set used'),
    ('Feature Importance',         '✅', 'SHAP values computed in Notebook 04'),
    ('Experiment Tracking',        '✅', 'All runs logged in MLflow'),
    ('Model Registry',             '✅', 'Best model registered & promoted to Production'),
    ('Data Drift Monitoring',      '✅', 'PSI computed for all key features'),
    ('Performance Monitoring',     '✅', 'AUC-ROC tracked on scoring population'),
    ('Batch Scoring Audit Trail',  '✅', 'ScoredAt + ModelVersion in gold table'),
    ('Bias / Fairness Testing',    '⚠️', 'Planned — add demographic parity checks'),
    ('Champion/Challenger Testing','⚠️', 'Planned — A/B model deployment'),
]

# Print the compliance checklist header for governance review
print('SR 11-7 Model Risk Management Checklist:')
print('─' * 65)
# Display each control area with its implementation status and follow-up notes
for item, status, note in checklist:
    print(f'{status}  {item:<35} {note}')
print('─' * 65)
